<a href="https://colab.research.google.com/github/ssykes-eth/ETH_275-0005-00L/blob/code_exercises/01_cx_multimodal_fraud_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Claim Triage: Reading Photos and Paperwork Together

**A hands-on companion to "Beyond Text RAG: Retrieval for Images, Documents, Tables, Audio, and Video"**

## The situation

You have just taken over **claims triage** at a mid-sized motor insurer. Every morning a queue of
new claims lands on your desk, and each one is a *pair*:

- a **photo** of the reported damage, sent in by the policyholder, and
- a **PDF claim form** — a two-column table of policy details, incident details, what's damaged,
  and how much the repair will cost.

Your team cannot investigate everything. Investigating a claim costs roughly what a small claim
pays out, so investigating everything would bankrupt the department — and investigating nothing
means paying every fraudulent claim in full. Your job is to decide, for each incoming claim:
**fast-track it, or refer it to an investigator?**

You want a system that helps you triage *and* shows its reasoning, because "the model said so"
is not something you can put in front of a regulator, an adjuster, or an unhappy customer.

Over this notebook you will build exactly that, and answer four practical questions:

1. **What does the paperwork actually say?** — turning a PDF of tables into fields you can compute with.
2. **What does the photo actually show?** — getting a machine to read damage severity from an image.
3. **Which past claims does this one resemble?** — retrieval, and why *what you index* decides whether it works.
4. **Should this claim be referred?** — combining the signals into a decision you can defend in one sentence.

> **Dummy data disclaimer:** every claim, name, policy number, and photo in this dataset is
> synthetic and fabricated for teaching purposes. Do not use this pipeline, as-is, on real claims.


In [ ]:
#@title 🗺️ Roadmap — the five sections ahead (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display
import uuid

def _roadmap():
    """Render the notebook roadmap as a horizontal stepper of gradient icons.

    Colours are inline here rather than from the shared palette, because this cell
    deliberately runs before Setup so the reader sees the map first.
    """
    uid = uuid.uuid4().hex[:8]
    steps = [
        ("📄", "Text only", "What plain extraction throws away"),
        ("🧩", "Parse the form", "Tables → fields you can compute with"),
        ("🖼️", "One shared space", "Pictures and words, same ruler"),
        ("🔎", "Retrieve precedent", "What you index decides what you find"),
        ("⚖️", "Triage a new claim", "Flags, a decision, a defensible sentence"),
    ]
    cards = "".join(f"""
      <div style="flex:1 1 150px;min-width:150px;text-align:center;">
        <div style="width:54px;height:54px;margin:0 auto 10px;border-radius:50%;
                    background:linear-gradient(135deg,#667eea,#764ba2);color:#fff;
                    font-size:24px;line-height:54px;">{icon}</div>
        <div style="font-weight:650;color:#2d2f45;font-size:14px;">{i}. {title}</div>
        <div style="color:#6a6d85;font-size:12px;margin-top:4px;line-height:1.35;">{sub}</div>
      </div>""" for i, (icon, title, sub) in enumerate(steps, 1))
    return HTML(f"""
    <div id="rm{uid}" style="font-family:system-ui,Segoe UI,Roboto,sans-serif;
         border-radius:18px;border:1px solid #ecebff;padding:24px 20px;
         background:linear-gradient(135deg,#f6f8ff,#fbf5ff);">
      <div style="font-weight:700;color:#2d2f45;margin-bottom:18px;font-size:15px;">
        Where we are going</div>
      <div style="display:flex;flex-wrap:wrap;gap:14px;">{cards}</div>
    </div>""")

display(_roadmap())


In [ ]:
#@title 📦 Install packages (double-click to view the code) { display-mode: "form" }
# Colab already ships torch, pandas, matplotlib and Pillow, so we install only what is
# genuinely absent -- normally just pdfplumber.
#
# Why the Pillow pin below: letting pip swap Pillow out underneath a running kernel
# leaves a half-replaced package on disk, and the next import dies with
# "cannot import name '_Ink' from 'PIL._typing'". pdfplumber depends on Pillow, so
# installing it can trigger exactly that. Pinning Pillow to the version already
# running keeps pip from touching it.
import importlib
import subprocess
import sys


def _is_missing(module_name):
    """True when the module cannot be imported in this runtime."""
    try:
        importlib.import_module(module_name)
        return False
    except Exception:
        return True


_needed = [pkg for pkg, mod in (("pdfplumber", "pdfplumber"),
                                ("transformers", "transformers"))
           if _is_missing(mod)]

if _needed:
    try:
        import PIL
        _guard = [f"pillow=={PIL.__version__}"]
    except Exception:
        _guard = []
    _r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *_guard, *_needed],
        check=False, capture_output=True, text=True,
    )
    if _r.returncode == 0:
        print("Installed: " + ", ".join(_needed))
    else:
        print("pip reported a problem:\n" + (_r.stderr or "")[-600:])
else:
    print("All required packages are already available.")

print("\nIf the next cell raises an ImportError from PIL, choose "
      "Runtime ▸ Restart session and run again from the top.")


In [ ]:
#@title ⚙️ Setup — imports, styling, palette { display-mode: "form" }
import warnings
warnings.filterwarnings("ignore")

import hashlib
import math
import os
import uuid
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pdfplumber
import torch
from IPython.display import HTML, display
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# House plot style
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 9

# Shared palette — reused by every plot and every HTML panel below
PURPLE, PURPLE2 = "#667eea", "#764ba2"      # primary / gradient accent
GREEN, RED, AMBER = "#39b36a", "#e0796d", "#e0a23c"   # good / bad / caution

# The only two label strings used anywhere in this notebook
LABELS = ("fraud", "non_fraud")
LABEL_COLOR = {"fraud": RED, "non_fraud": GREEN}

pd.set_option("display.max_colwidth", 90)
print("Setup complete.")


## Getting the data

Run the cell below to load the claim book — each claim a photograph paired with a PDF claim form.
It takes a few seconds the first time.


In [ ]:
#@title 📥 Load the claims data (double-click to view the code) { display-mode: "form" }
REPO = "eth-fdd-fs26/FDD-WE5-private"
CLONE_DIR = "FDD-WE5-private"
CANDIDATE_DIRS = (
    f"{CLONE_DIR}/Data/Multi_modal",
    "Data/Multi_modal",
    "Data",
)

# Set to True only if you intend to hand-upload a zip instead. Left False so the
# cell can never sit waiting on a file picker that nobody is watching.
ALLOW_MANUAL_UPLOAD = False


def find_data_dir(candidates=CANDIDATE_DIRS):
    """Return the first candidate folder holding images/, claims/ and labels.csv.

    Args:
        candidates: Relative paths to test, in priority order.

    Returns:
        The matching path as a string, or None if no candidate is complete.
    """
    for c in candidates:
        if (os.path.isdir(os.path.join(c, "images"))
                and os.path.isdir(os.path.join(c, "claims"))
                and os.path.isfile(os.path.join(c, "labels.csv"))):
            return c
    return None


def _credential():
    """Resolve the access credential, reporting which source supplied it.

    Returns:
        (value, source_description). value is None when nothing was found.
    """
    try:
        from google.colab import userdata
        value = userdata.get("GITHUB_TOKEN")
        if value and value.strip():
            return value.strip(), "Colab secret"
    except Exception as exc:                       # secret absent, or access not granted
        globals()["_secret_error"] = f"{type(exc).__name__}: {exc}"
    if os.environ.get("GITHUB_TOKEN", "").strip():
        return os.environ["GITHUB_TOKEN"].strip(), "environment variable"
    return None, "not found"


def _run_git(args, credential):
    """Run a git command that can never block on an interactive prompt.

    Returns:
        (ok, message) with the credential redacted out of the message.
    """
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="never")
    try:
        r = subprocess.run(args, env=env, capture_output=True, text=True, timeout=90)
    except subprocess.TimeoutExpired:
        return False, "timed out after 90s (network blocked, or the credential is wrong)"
    out = ((r.stderr or "") + (r.stdout or "")).strip()
    if credential:
        out = out.replace(credential, "***")
    return r.returncode == 0, out


DATA_DIR = find_data_dir()
_problems = []

if DATA_DIR is None:
    _cred, _source = _credential()

    if _cred is None:
        _problems.append(
            "No access credential available (checked Colab secrets and the environment)."
        )
        if "_secret_error" in globals():
            _problems.append(f"  Colab secret lookup said: {_secret_error}")
    else:
        print(f"Fetching the claim book… (credential from {_source})")
        if os.path.isdir(CLONE_DIR):
            ok, msg = _run_git(["git", "-C", CLONE_DIR, "pull", "--quiet"], _cred)
        else:
            ok, msg = _run_git(
                ["git", "clone", "--depth", "1", "--quiet",
                 f"https://{_cred}@github.com/{REPO}.git", CLONE_DIR], _cred)
        if not ok:
            _problems.append(f"git failed: {msg.splitlines()[-1] if msg else 'no output'}")
        else:
            DATA_DIR = find_data_dir()
            if DATA_DIR is None:
                inside = sorted(os.listdir(CLONE_DIR))[:8] if os.path.isdir(CLONE_DIR) else []
                _problems.append(
                    f"Fetch succeeded but no data folder was found inside. Saw: {inside}")

if DATA_DIR is None and ALLOW_MANUAL_UPLOAD:
    from google.colab import files
    print("Upload a zip containing images/, claims/ and labels.csv.")
    _up = files.upload()
    with zipfile.ZipFile(next(iter(_up)), "r") as z:
        z.extractall(".")
    DATA_DIR = find_data_dir()

if DATA_DIR is None:
    raise RuntimeError(
        "Could not load the claim book.\n\n"
        + "\n".join(f"  • {p}" for p in _problems)
        + "\n\nTo fix: open the 🔑 panel in the left sidebar, add a secret named "
          "GITHUB_TOKEN holding a GitHub personal access token with read access to "
          f"{REPO}, and switch on 'Notebook access' for it. Then re-run this cell.\n"
          "Alternatively set ALLOW_MANUAL_UPLOAD = True at the top of this cell to "
          "upload a zip by hand."
    )

print(f"Loaded the claim book from: {DATA_DIR}")


### Check the manifest before trusting it

`labels.csv` says which photo and which PDF belong to each claim. Before computing anything we check
it against what is actually on disk: every row has a label, every file it names exists, and no two
claims share the same photo.


In [ ]:
labels_df = pd.read_csv(os.path.join(DATA_DIR, "labels.csv"))
n_raw = len(labels_df)


def _row_is_usable(row):
    """True when the row has a label and both of its files exist on disk."""
    return (
        isinstance(row["label"], str)
        and os.path.isfile(os.path.join(DATA_DIR, "images", str(row["image_file"])))
        and os.path.isfile(os.path.join(DATA_DIR, "claims", str(row["pdf_file"])))
    )


usable = labels_df.apply(_row_is_usable, axis=1)
if not usable.all():
    dropped = list(labels_df.loc[~usable, "claim_id"])
    print(f"⚠️  Dropping {len(dropped)} row(s) with a missing label or missing file: {dropped}")
labels_df = labels_df[usable].reset_index(drop=True)

# Normalise label spelling once, so "non-fraud" and "non_fraud" can never disagree
# with the strings our decision functions produce.
labels_df["label"] = (labels_df["label"].str.strip().str.lower()
                      .str.replace("-", "_", regex=False))
unexpected = set(labels_df["label"]) - set(LABELS)
assert not unexpected, f"labels.csv contains unexpected label(s): {unexpected}"

# Duplicate-photo check
digests = {}
for _, row in labels_df.iterrows():
    with open(os.path.join(DATA_DIR, "images", row["image_file"]), "rb") as fh:
        digests.setdefault(hashlib.md5(fh.read()).hexdigest(), []).append(row["claim_id"])
duplicates = [ids for ids in digests.values() if len(ids) > 1]
if duplicates:
    print("⚠️  DUPLICATE PHOTOS — these claims share a byte-identical image:")
    for ids in duplicates:
        print(f"      {ids}")
else:
    print("✅ No duplicate photos.")

counts = labels_df["label"].value_counts()
majority = counts.max() / len(labels_df)
print(f"✅ {len(labels_df)} of {n_raw} claims usable — "
      + ", ".join(f"{k}: {v}" for k, v in counts.items()))
print(f"\nBASELINE TO BEAT: always guessing '{counts.idxmax()}' scores {majority:.0%}.")
labels_df


Keep that baseline in view. Every accuracy figure below is reported next to it, because an accuracy
that merely matches the baseline is worth nothing — the model would be adding no information at all.


### Meet the claims

Before any modelling, look at the data. Here is every claim in the book — photo, claim id, and
the outcome that was eventually established. Green borders are legitimate claims, red are the ones
later confirmed as fraud.

Spend a moment on this. Ask yourself whether *you* could sort these into two piles from the
photographs alone — the answer to that question shapes everything that follows.


In [ ]:
#@title 📊 Visualization: the whole claim book (double-click to view the code) { display-mode: "form" }
n = len(labels_df)
cols = 5
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 2.7 * rows))
axes = np.atleast_1d(axes).ravel()

for ax, (_, row) in zip(axes, labels_df.iterrows()):
    img = Image.open(os.path.join(DATA_DIR, "images", row["image_file"])).convert("RGB")
    ax.imshow(img)
    ax.set_xticks([]); ax.set_yticks([])
    colour = LABEL_COLOR[row["label"]]
    for side in ax.spines.values():
        side.set_visible(True); side.set_color(colour); side.set_linewidth(3)
    ax.set_title(f"{row['claim_id']}\n{row['label']}", fontsize=8, color=colour, pad=4)

for ax in axes[n:]:
    ax.axis("off")

plt.suptitle("Every claim in the book — border colour is the confirmed outcome",
             fontsize=11, y=1.005)
plt.tight_layout()
plt.show()


Two things are worth noticing already.

**The photographs do not sort themselves.** Wrecked cars appear on both sides of the ledger, and
so do minor scuffs. There is no "fraud look". If a photo alone were enough, claims triage would
not be a job.

**But some pairs will turn out to be contradictions.** A few of these photos, once you read the
form that came with them, describe a completely different event from the one the policyholder
wrote down. Finding those is where the second modality earns its place, and it is what Part 5
is built to catch.


---

## Part 1 — How you process the data decides what you can ask

Before any model, any embedding, any retrieval: **something has to turn the document into data.**
That step is usually treated as plumbing and skipped over. It is not plumbing. It silently fixes the
ceiling on everything built above it — a question the ingestion step throws away is a question no
model downstream can ever answer.

To make that concrete, here is one question a triage lead asks constantly:

> **Is this repair estimate more than a third of the policy's total cover?**

Hold that question in mind. Below is the claim form processed the way a plain text-RAG pipeline
would do it — extract all the words, embed them, move on. Both numbers you need are in there
somewhere. See if you could reliably pull them out.


In [ ]:
sample_id = labels_df.iloc[0]["claim_id"]
sample_pdf = os.path.join(DATA_DIR, "claims", labels_df.iloc[0]["pdf_file"])

with pdfplumber.open(sample_pdf) as pdf:
    raw_text = pdf.pages[0].extract_text()

print(f"--- {sample_id}.pdf, read as plain text ---\n")
print(raw_text)


Try to answer the question from that.

`$25,000` and `$12,850` are both present — along with `$18,500`, `98,452`, and `2021`. **Nothing marks
which number is which.** The form was laid out as *two side-by-side tables*, and flattening it to a
line-by-line stream has interleaved them: `Policy Reference Number` got split across two lines, and
the incident description is chopped into fragments alternating with unrelated fields, so
*"While driving on Highway 47"* now sits next to *"Time of Loss"*.

You cannot compute `repair ÷ cover` from this, because you cannot reliably say which number is the
repair and which is the cover.

**And in a real RAG system it gets worse, because of chunking.** Documents are not embedded whole —
they are split into chunks of a few hundred tokens. On a two-column table a chunk boundary falls
wherever it falls, so `Estimated Repair Cost` can land in one chunk and `$12,850` in the next. The
retriever then returns a passage containing a number with no idea what that number *is* — and the
model, asked for the repair cost, will cite it anyway. That is the mechanism behind a great many
confident, wrong RAG answers.

The fix is not a better model or a bigger vector database. It is **processing the document as the
structured thing it already is** — which is Part 2.


---

## Part 2 — Processing the form as structure, not prose

Same PDF, different processing. The forms are ruled tables: the file carries the lines and rectangles
that draw the grid, and that geometry says which cell sits in which column. Plain text extraction
throws the geometry away. A table-aware reader keeps it.

General-purpose parsers such as `docling` (mentioned in the lecture) go to considerable lengths to
recover that geometry on documents they have never seen; the optional section at the end of this part
walks through how. Our forms are far easier than the general case, so `pdfplumber`'s table detector
gets us most of the way — it hands back each table as a grid of cells, and we walk each row pairing
up `(label, value)`.


In [ ]:
def extract_claim_fields(pdf_path):
    """Parse a claim PDF's tables into a flat {field_label: value} dictionary.

    The form is a grid of (label, value) cells, often two side-by-side pairs per row
    (e.g. "Policy Holder Name | John Smith | Claim Registration Date | 09-Aug-2026"),
    with empty spacer columns and section-header rows that have a label but no value.
    We scan each row left to right and greedily pair non-empty cells.

    Args:
        pdf_path: Path to the claim PDF.

    Returns:
        A dict of field label to value. Empty if the document has no extractable text.
    """
    fields = {}
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    cells = [c.strip().replace("\n", " ") if isinstance(c, str) else None
                             for c in row]
                    i = 0
                    while i < len(cells) - 1:
                        label, value = cells[i], cells[i + 1]
                        if label and value:
                            fields[label] = value
                            i += 2
                        else:
                            i += 1
    return fields


sample_fields = extract_claim_fields(sample_pdf)
print(f"{len(sample_fields)} fields recovered from {sample_id}.pdf\n")
pd.DataFrame(sample_fields.items(), columns=["field", "value"]).head(12)


### The same document, both ways

Now put the two processings next to each other, and ask the Part 1 question again.


In [ ]:
#@title 📊 Visualization: text-only vs table-aware processing (double-click to view the code) { display-mode: "form" }
with pdfplumber.open(sample_pdf) as pdf:
    _raw = (pdf.pages[0].extract_text() or "").splitlines()

_left = [ln for ln in _raw if ln.strip()][:14]
_keys = ["Policy Holder Name", "Sum Insured", "Damage Claimed",
         "Estimated Repair Cost", "Police Report Number", "Third Party Involved"]
_right = [(k, sample_fields.get(k, "—")) for k in _keys]

print("TEXT-ONLY PROCESSING                              TABLE-AWARE PROCESSING")
print("(a stream of words — which number is which?)      (fields you can compute with)")
print("-" * 52 + "  " + "-" * 46)
for i in range(max(len(_left), len(_right))):
    lhs = _left[i][:50] if i < len(_left) else ""
    rhs = f"{_right[i][0][:26]:26s} = {_right[i][1][:18]}" if i < len(_right) else ""
    print(f"{lhs:52s}  {rhs}")

# The Part 1 question, now answerable in one line.
def _as_number(text):
    """Turn a currency string like '$12,850' into a float."""
    return float(str(text).replace("$", "").replace(",", "").strip())


_cost = _as_number(sample_fields["Estimated Repair Cost"])
_cover = _as_number(sample_fields["Sum Insured"])
print("\n" + "=" * 100)
print(f"Q: is the repair estimate more than a third of the cover?")
print(f"A: {_cost:,.0f} / {_cover:,.0f} = {_cost / _cover:.0%}  →  "
      f"{'YES' if _cost / _cover > 1 / 3 else 'no'}")
print("Unanswerable from the left column. One division on the right.")


That is the whole argument for taking ingestion seriously. Same PDF, same information physically
present in both columns — but only one of them lets you *ask a question of it*.

Note what actually changed: not the model, not the retriever, not the embedding. **The processing.**
Keep that in mind at Part 4, where the identical point reappears in a different costume.


### Exercise 1. Catch a parser that failed quietly

`extract_claim_fields` never raises. Hand it a PDF it cannot read and it returns an empty dictionary;
hand it one whose columns are misaligned and it returns confident nonsense. Either way the notebook
carries on, and the damage shows up much later as a number that is quietly wrong.

So every parser needs a validator sitting behind it. Ours looks for the two failures that actually
occur with documents like these:

1. **Nothing came back at all** — the page has no extractable text.
2. **A value is itself a field label** — a sure sign the columns are shifted, because
   `Policy Holder Name` should never have `Sum Insured` as its *value*.

**Task.** Complete `validate_fields` so it detects both, appending one message per problem to
`problems`. The third check — required fields missing — is already written for you.


In [ ]:
# Every field label these forms are known to use. A value matching one of these
# is a symptom, not a value.
KNOWN_LABELS = {
    "Policy Reference Number", "Claim Reference Number", "Policy Holder Name",
    "Claim Registration Date", "Policy Inception Date", "Service Provider",
    "Policy Expiry Date", "Sum Insured", "Asset Category", "Balance Sum Insured",
    "Manufacturer", "Vehicle Registration", "Model", "VIN Number", "Purchase Date",
    "Odometer Reading", "Date of Loss", "Time of Loss", "Weather Conditions",
    "Road Conditions", "Vehicle Speed", "Police Report Number", "Third Party Involved",
    "Third Party Information Available", "Injuries Reported", "Damage Claimed",
    "Estimated Repair Cost", "Vehicle Operable After Incident", "Towed From Scene",
    "Description of Incident",
}


#### Your turn


In [ ]:
# 🎯 YOUR TURN — Exercise 1: catch a parser that failed quietly.
#
# 💭 Think first: this function returns a *list of problems*, and an empty list means
#    "looks fine". Why is returning nothing-is-wrong safer than returning True/False?
#    (Hint: what does the caller get to print?)

def validate_fields(claim_id, fields):
    """Check parsed output for the two ways this parser fails silently.

    Args:
        claim_id: Identifier, used in the returned message.
        fields: The dict returned by extract_claim_fields.

    Returns:
        A list of human-readable problem strings. Empty means the parse looks sound.
    """
    problems = []

    # 🎯 Implement (1): fire when `fields` came back empty.
    #    Hint: an empty dict is falsy in Python, so `not fields` is True when it is empty.
    if ...:
        problems.append(f"{claim_id}: NO FIELDS AT ALL — the page has no extractable text")
        return problems

    # 🎯 Implement (2): keep the (label, value) pairs whose VALUE is itself a field label.
    #    Hint: you are filtering `fields.items()`; .
    shifted = [(k, v) for k, v in fields.items() if ...]
    if shifted:
        problems.append(f"{claim_id}: value is itself a field label {shifted[:2]} "
                        "— the columns are misaligned")

    # 3. Written for you: the fields everything downstream depends on.
    for required in ("Damage Claimed", "Estimated Repair Cost", "Sum Insured"):
        if required not in fields:
            problems.append(f"{claim_id}: missing required field '{required}'")
    return problems


In [ ]:
# ✅ Self-check — run this straight after your answer.
_healthy = {"Damage Claimed": "front bumper", "Estimated Repair Cost": "$1,000",
            "Sum Insured": "$20,000"}
_shifted = {**_healthy, "Policy Holder Name": "Sum Insured"}

assert validate_fields("t", {}), "check 1: an empty dict should report a problem"
assert "NO FIELDS" in validate_fields("t", {})[0], "check 1: should report the empty-page message"
assert validate_fields("t", _shifted), "check 2: a value equal to a field label should be caught"
assert not validate_fields("t", _healthy), \
    f"a healthy parse should report nothing, but got {validate_fields('t', _healthy)}"
print("✅ Exercise 1 looks right.")


In [ ]:
parsed, parse_problems = {}, []
for _, row in labels_df.iterrows():
    f = extract_claim_fields(os.path.join(DATA_DIR, "claims", row["pdf_file"]))
    parsed[row["claim_id"]] = f
    parse_problems.extend(validate_fields(row["claim_id"], f))

if parse_problems:
    print("⚠️  PARSER PROBLEMS FOUND:\n")
    for p in parse_problems:
        print("   ", p)
else:
    print("✅ All documents parsed cleanly.")

ok_ids = [cid for cid, f in parsed.items() if not validate_fields(cid, f)]
print(f"\n{len(ok_ids)} of {len(parsed)} claims are usable downstream.")


**Reading the result.** The check either passes for every document or names the ones it cannot
vouch for — and either way we know, rather than finding out later via a number that quietly went
wrong.

That matters because document parsing fails *silently*. It does not raise; it returns something
plausible-looking and carries on. Anything the validator rejects is excluded from the results below
and reported, so the claim count you see is always the count actually used.


---

### ⭐ Bonus — would this parser work on other documents?

*Optional. Everything the notebook needs is already built; skip ahead to Part 3 if you are short of
time. It is here because "does this generalise?" is the right question to ask of any demo, including
this one.*

**No, not as written — and it is worth being precise about why.**

`extract_tables()` is the general part: it will find the grid in any PDF that has real text and ruled
lines. The rule we built on top of it is not. *Take adjacent cells left to right as (label, value)*
encodes an assumption about **this kind of document** — a key-value form, where a field's meaning sits
immediately to its left. Four assumptions, in fact, all true of these claim forms and none guaranteed
elsewhere:

1. **Meaning runs left-to-right.** A value belongs to the cell beside it, not to a column header above
   it. Point the same rule at an ordinary data table — dates down one column, amounts down another —
   and it cheerfully pairs `Date` with `Description`, then `02-Aug-2026` with `Tow service`, and
   carries on. No error, no warning, just plausible-looking nonsense handed to everything downstream.
2. **Labels are unique.** We store them in a flat dictionary, so a form repeating "Date" in two
   sections silently keeps only the last one.
3. **The page has real text and ruled lines.** No scans, no whitespace-aligned columns, no document
   whose text was flattened into vector outlines on export.
4. **The field names are known exactly.** Everything downstream asks for `"Estimated Repair Cost"` by
   string, so another insurer's form would parse perfectly and then produce an empty record.

**And that is a perfectly respectable place to land.** This is a *specialised* pipeline: one document
type, one layout, one known set of fields — with validation that fails loudly when those assumptions
are violated. A great many production document systems are exactly this, and they work *because* the
scope is narrow and somebody checked the assumptions. A pipeline that handles one form well and
refuses everything else is far more useful than one that handles everything badly and tells you
nothing.

The cost is that it does not transfer. The moment the documents are heterogeneous — contracts,
letters, invoices from forty different suppliers — you need something else.


#### So what does handle documents you have not seen?

Our forms are ruled tables with a text layer — close to the easy case. Since most documents are not,
here is what a general pipeline does. Knowing the stages is what lets you ask a vendor a useful
question.

A PDF is a **drawing format, not a document format.** It stores "put glyph 'A' at x=72, y=310 in 11pt
Helvetica." There is no paragraph, no table, no reading order — a human infers those from position.
Every parser is reconstructing structure that was discarded at export — and some export routes
discard more than others. "Print to PDF", for instance, can convert every glyph into a vector outline,
leaving a page that looks perfect and contains no text at all.

**1 · Get characters with coordinates.** Born-digital files give this up directly (`pdfminer.six`,
`pdfplumber`, `PyMuPDF`). Scans do not, so they are rasterised and passed through OCR (Tesseract,
PaddleOCR, or a cloud service). You cannot tell the two apart from the filename, so real pipelines
*test for a text layer and branch* — the same check as our validation cell.

**2 · Reconstruct layout.** Characters → words → lines → blocks, then **reading order**, which is what
stops a two-column page from being read straight across. Either geometric rules (fast, deterministic,
brittle — the family our pairing rule belongs to) or a **layout model**: an object detector run over
the page image that labels regions as Title / Paragraph / Table / Figure. The learned version is what
survives a layout it has not seen, and it is the main thing a document-AI product is selling.

**3 · Recover table structure.** Detecting a table is easy; recovering its cells is not. *Ruled*
tables can use the drawn lines — that is our case, and why something simple works here. Tables with
no rulings need the columns inferred, or a learned model such as Table Transformer or **TableFormer**,
which is the table engine inside `docling`.

**4 · Turn regions into fields.** Layout-aware transformers (LayoutLMv3) encode text *plus* its 2D
position; OCR-free models (Donut) go from page image straight to JSON. Increasingly a **vision-language
model** is handed the page image and asked for JSON directly — which handles layouts nothing else can,
at the price of cost, non-determinism, no bounding boxes for provenance, and the risk it **invents a
plausible value**. A parser returning nothing is safe; a parser inventing `$12,850` is not.

Common production shape: deterministic extraction where the layout is known, a VLM only for pages that
fail validation.

#### Why this decides RAG quality

Parsing shows up downstream as retrieval quality, through chunking:

- **Chunk on element boundaries, not fixed token windows** — otherwise tables get cut in half.
- **Keep tables intact** and serialise them (markdown or HTML) so headers stay attached to values.
- **Carry metadata** — page, section, bounding box — so an answer can cite a location instead of
  asserting a number.
- **Prepend section context** to each chunk, so a chunk containing `$12,850` still knows it sits under
  *Estimated Repair Cost*.

Blunt version: in most enterprise RAG systems more answer quality is won or lost in stages 2–3 than by
any choice of embedding model or vector database. Ingestion is not plumbing.


---

## Part 3 — One space, two kinds of evidence

We can now read the paperwork. The photograph is still an image, and we need a way to put the two
side by side.

**CLIP** does that. It is two encoders trained together on hundreds of millions of (image, caption)
pairs scraped from the web:

- an **image encoder** — picture → vector
- a **text encoder** — sentence → vector

They were trained so that a picture scores higher against its own caption than against somebody
else's. What that buys us is specific, and worth stating precisely because it is easy to overclaim:
**a set of candidate captions can be ranked against a photograph, and the same captions can be ranked
against a sentence.** One yardstick, two kinds of evidence.


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

# 🎯 YOUR TURN — Load a pretrained CLIP model for classification.
# 💭 Think first: In sklearn, you'd write something like:
#    `model = LogisticRegression()` then `model.fit(X, y)`
#    But CLIP is different — it's already "pretrained" on 400M image-text pairs.
#    So here, we just `.from_pretrained()` and it's ready to go. No `.fit()` needed.

clip_model =  ...  # Hint: CLIPModel.from_pretrained() 🎯 TODO
clip_processor = ...  # Hint: CLIPProcessor.from_pretrained() 🎯TODO

# CLIP's text encoder hard-truncates past 77 tokens. Everything we hand it here is a
# short caption or a one-line damage description, so we stay well inside that.

print(f"Loaded {CLIP_MODEL_NAME} on {DEVICE}")



In [ ]:
def _pooled(out):
    """Return the projected feature tensor from a get_*_features call.

    transformers < 5 returns a plain tensor; transformers >= 5 returns a
    BaseModelOutputWithPooling whose .pooler_output holds the projected features.
    Handling both keeps this notebook working on any Colab runtime.
    """
    return getattr(out, "pooler_output", out)


@torch.no_grad()
def embed_image(image):
    """Encode a PIL image into a unit-length CLIP vector.

    Args:
        image: A PIL.Image in RGB.

    Returns:
        A 1-D numpy array of length 512, normalised to length 1.
    """
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    feats = _pooled(clip_model.get_image_features(**inputs))
    return (feats / feats.norm(p=2, dim=-1, keepdim=True)).cpu().numpy()[0]


@torch.no_grad()
def embed_text(text):
    """Encode a short piece of text into a unit-length CLIP vector.

    Args:
        text: The string to encode (a short caption or damage description).

    Returns:
        A 1-D numpy array of length 512, normalised to length 1.
    """
    inputs = clip_processor(text=[text], return_tensors="pt",
                            padding=True, truncation=True).to(DEVICE)
    feats = _pooled(clip_model.get_text_features(**inputs))
    return (feats / feats.norm(p=2, dim=-1, keepdim=True)).cpu().numpy()[0]


print("Embedding helpers ready.")


### How a grading actually runs

Three steps, one line of code each.

| | | |
|---|---|---|
| **1 · Similarity** | dot product against each caption (all vectors are unit length) | $s_k = v \cdot t_k$ |
| **2 · Temperature** | rescale by CLIP's learned $\tau = \exp(\texttt{logit\_scale}) \approx 100$ | $z_k = \tau \, s_k$ |
| **3 · Softmax** | three scores become three probabilities | $p_k = e^{z_k} / \sum_j e^{z_j}$ |

**Where does $\tau$ come from?** Not from us. It is a *trained weight* sitting in the checkpoint
next to the encoder parameters. CLIP's authors made the temperature learnable, started it at
$1/0.07 \approx 14$, and capped it at 100 so it could not grow without bound — and training pushed it
straight to that ceiling. The released model stores `logit_scale = 4.6052`, and $e^{4.6052} = 100.0$.
We read it off the model rather than hard-coding it:

```python
LOGIT_SCALE = float(clip_model.logit_scale.exp().detach())
```

Step 2 is the one people skip, and it is why CLIP so often looks useless. Grading `claim_06`'s
photograph:

| | minor | moderate | severe |
|---|---|---|---|
| raw cosine | 0.285 | 0.299 | **0.313** |
| softmax, no $\tau$ | 33% | 33% | 34% |
| softmax, with $\tau$ | 4% | 18% | **77%** |

Identical inputs. Without the temperature the model looks like it has no opinion; with it, the answer
is unambiguous.

From $p$ we keep two things: $\arg\max_k p_k$ as a label, and $\sum_k k \, p_k$ as a 0–2 score that
preserves the uncertainty instead of discarding it.

### Why we compare labels, not numbers

Now grade the *form text* for the same claim — "front fender dent, minor paint scratches" — against
those same three captions:

| | minor | moderate | severe |
|---|---|---|---|
| **photo**, raw cosine | 0.285 | 0.299 | **0.313** |
| **form text**, raw cosine | **0.811** | 0.757 | 0.738 |

Read the *severe* column across: the form scores **0.738**, the photo **0.313**. Taken at face value
the paperwork is twice as severe as the picture — which is exactly backwards.

Image vectors and text vectors occupy **separate regions** of CLIP's space, so every text cosine sits
high and every image cosine sits low. The two rows are not on the same scale and never will be.

Step 3 is what makes this safe: the softmax only ever compares **the three captions against each
other, for one query at a time.** Add the same constant to all three and $p$ is unchanged — so the
offset cancels. After rescaling:

- photo → **severe** (77%)
- form → **minor** (100%)

**Rank within each modality, then compare the ranks.** That disagreement is the multimodal check in
Part 5 — two rankings, never two raw numbers.


### Exercise 2. Turn similarities into a verdict

`zero_shot` above already does the first two steps: it takes the dot product against each caption and
multiplies by the temperature, then hands back the probabilities. What is left is reading an answer
out of those three numbers — and we want two different readings of them.

**A label**, for a human: whichever caption scored highest.

**A score**, for the retrieval index: the *expected value* $\sum_k k \, p_k$ with minor = 0,
moderate = 1, severe = 2. A confident "severe" gives ≈ 2, a confident "minor" ≈ 0, and a genuinely
uncertain photo lands in between — which keeps the model's hesitation instead of throwing it away.

**Task.** Complete `read_severity` so it returns both.


In [ ]:
SEVERITY_PROMPTS = {
    "minor":    "a photo of a car with minor cosmetic damage, a small scratch dent or chip",
    "moderate": "a photo of a car with moderate damage, dents and scrapes on body panels",
    "severe":   "a photo of a car with severe structural damage, crushed wrecked and undrivable",
}
SEVERITY_LEVELS = list(SEVERITY_PROMPTS)
SEVERITY_EMB = np.array([embed_text(p) for p in SEVERITY_PROMPTS.values()])
LOGIT_SCALE = float(clip_model.logit_scale.exp().detach())


def zero_shot(query_vec, prompt_embeddings):
    """Score one embedding against a set of candidate captions, CLIP-style.

    Args:
        query_vec: Unit-length embedding — an image OR a piece of text.
        prompt_embeddings: Array of unit-length caption embeddings, one row each.

    Returns:
        (raw_cosines, probabilities) — both 1-D arrays aligned with the captions.
    """
    sims = prompt_embeddings @ query_vec
    logits = sims * LOGIT_SCALE
    probs = np.exp(logits - logits.max())
    return sims, probs / probs.sum()


#### Your turn


In [ ]:
# 🎯 YOUR TURN — Exercise 2: turn three probabilities into a verdict.
#
# 💭 Think first: why keep the 0-2 score at all, when we already have a label? What
#    does "1.7" tell a retrieval index that the word "severe" does not?

def read_severity(vec):
    """Grade damage severity from either a photo embedding or a text embedding.

    The same function serves both modalities, because CLIP put them in the same
    space and we score both against the same three yardstick captions.

    Args:
        vec: A unit-length CLIP embedding of a photo or of a piece of text.

    Returns:
        (level, score, probabilities) where level is one of SEVERITY_LEVELS and
        score is a 0-2 number (0 = minor, 2 = severe).
    """
    _, probs = zero_shot(vec, SEVERITY_EMB)   # probs is [p_minor, p_moderate, p_severe]

    # 🎯 Implement `level`: the SEVERITY_LEVELS entry with the highest probability.
    #    Hint: `probs.argmax()` gives you the winning position (0, 1 or 2). Wrap it in
    #    int() and use it to index SEVERITY_LEVELS.
    level = ...

    # 🎯 Implement `score`: the expected value, i.e. 0*p_minor + 1*p_moderate + 2*p_severe.
    #    Hint: np.arange(3) is the array [0, 1, 2], and `probs @ np.arange(3)` is exactly
    #    that weighted sum.
    score = ...

    return level, float(score), probs


In [ ]:
# ✅ Self-check — run this straight after your answer.
# Grading the "severe" caption against itself must come out unambiguously severe.
try:
    _level, _score, _probs = read_severity(SEVERITY_EMB[2])
except TypeError as _e:
    raise AssertionError(
        "read_severity still has a `...` in it — fill in both `level` and `score`.") from _e
assert _level == "severe", f"the severe caption should grade as 'severe', got {_level!r}"
assert abs(_probs.sum() - 1) < 1e-6, "probabilities should sum to 1"
assert _score > 1.5, f"a confident 'severe' should score near 2, got {_score:.2f}"

_level_min, _score_min, _ = read_severity(SEVERITY_EMB[0])
assert _level_min == "minor", f"the minor caption should grade as 'minor', got {_level_min!r}"
assert _score_min < 0.5, f"a confident 'minor' should score near 0, got {_score_min:.2f}"
print("✅ Exercise 2 looks right.")


In [ ]:
# Grade one claim both ways, with the same three captions.
DEMO_ID = "claim_06" if "claim_06" in parsed else labels_df.iloc[0]["claim_id"]
demo_row = labels_df[labels_df.claim_id == DEMO_ID].iloc[0]
demo_img = Image.open(os.path.join(DATA_DIR, "images", demo_row["image_file"])).convert("RGB")
demo_text = parsed[DEMO_ID].get("Damage Claimed", "")


def show_grading(vec, what):
    """Print raw cosines next to the softmax, with and without the temperature."""
    sims, probs = zero_shot(vec, SEVERITY_EMB)
    naive = np.exp(sims - sims.max()); naive /= naive.sum()
    print(f"\n{what}")
    print(f"  {'':18s}" + "".join(f"{lvl:>11s}" for lvl in SEVERITY_LEVELS))
    print(f"  {'raw cosine':18s}" + "".join(f"{s:11.3f}" for s in sims))
    print(f"  {'softmax, no tau':18s}" + "".join(f"{p:10.0%} " for p in naive))
    print(f"  {'softmax, with tau':18s}" + "".join(f"{p:10.0%} " for p in probs))
    print(f"  -> reads as {SEVERITY_LEVELS[int(probs.argmax())].upper()}")


show_grading(embed_image(demo_img), f"{DEMO_ID} PHOTOGRAPH")
show_grading(embed_text(demo_text), f'{DEMO_ID} FORM TEXT — "{demo_text[:44]}"')
print("\nSame three captions, two kinds of evidence, two rankings — and they disagree.")
print("Note the raw cosines are on completely different scales: never compare them across rows.")


In [ ]:
#@title 📊 Visualization: photo vs form, graded on the same scale (double-click to view the code) { display-mode: "form" }
show_ids = list(labels_df["claim_id"])[:6]
fig, axes = plt.subplots(2, len(show_ids), figsize=(2.3 * len(show_ids), 5.4),
                         gridspec_kw={"height_ratios": [2.1, 1]})

for j, cid in enumerate(show_ids):
    row = labels_df[labels_df.claim_id == cid].iloc[0]
    img = Image.open(os.path.join(DATA_DIR, "images", row["image_file"])).convert("RGB")
    vec = embed_image(img)
    level, score, pr = read_severity(vec)

    axes[0, j].imshow(img)
    axes[0, j].set_xticks([]); axes[0, j].set_yticks([])
    axes[0, j].set_title(f"{cid}\nphoto reads: {level}", fontsize=8, pad=4)

    axes[1, j].bar(SEVERITY_LEVELS, pr, color=[GREEN, AMBER, RED], width=0.65)
    axes[1, j].set_ylim(0, 1)
    axes[1, j].set_xticklabels(SEVERITY_LEVELS, rotation=35, ha="right", fontsize=7)
    axes[1, j].tick_params(axis="y", labelsize=7)
    if j == 0:
        axes[1, j].set_ylabel("zero-shot\nprobability", fontsize=8)

plt.suptitle("CLIP grading damage severity from the photograph alone — no training required",
             fontsize=11, y=1.0)
plt.tight_layout()
plt.show()


**Reading the result.** CLIP grades damage severity sensibly, straight out of the box, with no
labelled training data of ours. That is genuinely useful: we now have a number describing *what the
photograph shows*, which we can set against *what the form claims*.

Be clear about what this is not, though. CLIP is judging **how bad the damage looks** — it has no
concept of fraud, and nothing in its training taught it what a suspicious claim is. Severity is an
ingredient. On its own it is not a verdict.


---

## Part 4 — Retrieval: what you index decides what you can find

Now the RAG part. Given a new claim, we want to pull up the historical claims most like it, so a
decision can cite precedent instead of asserting a score.

The question nobody asks carefully enough is **what to index**. It is tempting to embed the photo
and the form with CLIP, glue the two vectors together, and search that. It runs, it looks
sophisticated — and the measurement below shows it carries almost no information about the outcome.
CLIP similarity asks whether two claims *look and read alike*, and every claim here is a photo of a
damaged car attached to an insurance form. They all look alike.

The question an adjuster actually asks is different: **"which past claims share this one's risk
structure?"** That lives in the fields we just parsed — how big the claim is relative to the cover,
whether anyone can corroborate it, whether the car was driveable — plus the severity CLIP read from
the photo. So that is what we index.


### What exactly goes into the index

Not the photo, not the form text — **eleven numbers per claim**, each one a plain attribute of the
claim. Nothing here is a fraud rule; the retriever is left to work out which combinations matter.

| # | feature | where it comes from | what it captures |
|---|---|---|---|
| | | **how much money is at stake** | |
| 1 | `log_cost` | `Estimated Repair Cost` | size of the claim, on a log scale so $500 and $50,000 are comparably spaced |
| 2 | `cost_ratio` | cost ÷ `Sum Insured` | how big the ask is *relative to the policy* |
| | | **how violent the event was** | |
| 3 | `speed` | `Vehicle Speed` | mph, parsed out of "Approximately 45 mph" |
| 4 | `not_operable` | `Vehicle Operable After Incident` | 1 if the car could not be driven away |
| 5 | `towed` | `Towed From Scene` | 1 if it had to be recovered |
| | | **who can corroborate the story** | |
| 6 | `third_party_yes` | `Third Party Involved` | another vehicle is named |
| 7 | `third_party_unknown` | `Third Party Involved` | another vehicle is blamed but unidentified |
| 8 | `tp_info_missing` | `Third Party Information Available` | no contact details were taken |
| 9 | `police_filed` | `Police Report Number` | a report number exists |
| 10 | `police_not_filed` | `Police Report Number` | expected but never filed |
| | | **what the photograph shows** | |
| 11 | `photo_severity` | CLIP zero-shot (Part 3) | 0 = minor … 2 = severe, the expected value from the softmax |

Ten come from the parsed PDF, one from the image. Note that features 6–10 are *indicators* — a
categorical field expanded into separate 0/1 columns, because "unknown" is not halfway between "yes"
and "no" and encoding it as 0.5 would invent an ordering that does not exist.


### Exercise 3. Decide what the index knows

This is the function that decides what retrieval can *possibly* find. Anything not encoded here is
invisible to the search, no matter how good the similarity maths is.

Seven of the eleven features are written for you. Four are left, chosen because each is a different
kind of encoding: a **ratio**, a plain **indicator**, an indicator with a **subtlety**, and the one
value that arrives from the **image** rather than the document.

**Task.** Fill in the four missing entries in the returned dictionary.


In [ ]:
def money(value):
    """Parse a currency string like '$12,850' into a float. Returns None if unparseable."""
    try:
        return float(str(value).replace("$", "").replace(",", "").strip())
    except (ValueError, AttributeError, TypeError):
        return None


def speed_mph(value):
    """Pull the numeric mph out of strings like 'Approximately 45 mph' or '0 mph (Parked)'."""
    digits = "".join(ch for ch in str(value) if ch.isdigit() or ch == ".")
    return float(digits) if digits else 0.0


# The index schema, declared in one place so it is obvious what is being searched.
FEATURE_NAMES = [
    "log_cost", "cost_ratio",                                       # money at stake
    "speed", "not_operable", "towed",                               # violence of the event
    "third_party_yes", "third_party_unknown", "tp_info_missing",    # corroboration
    "police_filed", "police_not_filed",
    "photo_severity",                                               # the image
]


#### Your turn


In [ ]:
# 🎯 YOUR TURN — Exercise 3: decide what the index knows.
#
# 💭 Think first: `police_not_filed` is not simply "no report number". A form saying
#    "Not Required" is an honest small claim; one saying "Not Filed" is a choice
#    somebody made. Why must those two be different numbers rather than both zero?

def build_features(fields, photo_severity):
    """Turn one claim's parsed fields plus its photo reading into the index vector.

    Args:
        fields: Parsed {label: value} dict for the claim.
        photo_severity: 0-2 damage score from read_severity on the photograph.

    Returns:
        Dict keyed by FEATURE_NAMES, every value a float.
    """
    cost = money(fields.get("Estimated Repair Cost")) or 0.0
    cover = money(fields.get("Sum Insured")) or 1.0
    police = str(fields.get("Police Report Number", "")).strip().lower()
    tp_involved = str(fields.get("Third Party Involved", "")).strip().lower()
    tp_info = str(fields.get("Third Party Information Available", "")).strip().lower()
    has_report = police.startswith("pr-")          # True when a real "PR-..." number is present

    return {
        "log_cost":            math.log10(cost + 1),

        # 🎯 (1) How big is the repair *relative to the policy*?
        #        Hint: you already have `cost` and `cover`. One division.
        "cost_ratio":          ...,

        "speed":               speed_mph(fields.get("Vehicle Speed")),
        "not_operable":        float(fields.get("Vehicle Operable After Incident") == "No"),
        "towed":               float(fields.get("Towed From Scene") == "Yes"),
        "third_party_yes":     float(tp_involved == "yes"),
        "third_party_unknown": float(tp_involved == "unknown"),

        # 🎯 (2) 1.0 when no third-party contact details were taken, else 0.0.
        #        Hint: `tp_info` is already lower-cased, so compare it to "no".
        #        Follow the pattern of the two lines just above: float(<condition>)
        "tp_info_missing":     ...,

        "police_filed":        float(has_report),

        # 🎯 (3) 1.0 when a report was expected and never filed, else 0.0.
        #        Hint: that means there is NO report number AND the form does not say
        #        "not required". Both pieces are ready: `has_report` and `police`.
        "police_not_filed":    ...,

        # 🎯 (4) The damage severity read off the photograph.
        #        Hint: it was handed to this function as an argument — just pass it through.
        "photo_severity":      ...,
    }


In [ ]:
# ✅ Self-check — run this straight after your answer.
_probe = {"Estimated Repair Cost": "$10,000", "Sum Insured": "$40,000",
          "Third Party Information Available": "No", "Police Report Number": "Not Filed"}
_f = build_features(_probe, photo_severity=1.5)

_unfilled = [k for k, v in _f.items() if v is ...]
assert not _unfilled, f"still to fill in: {_unfilled}"

assert abs(_f["cost_ratio"] - 0.25) < 1e-9, \
    f"(1) $10,000 of $40,000 cover is 0.25, got {_f['cost_ratio']}"
assert _f["tp_info_missing"] == 1.0, "(2) should be 1.0 when no contact details were taken"
assert build_features({**_probe, "Third Party Information Available": "Yes"},
                      1.5)["tp_info_missing"] == 0.0, "(2) should be 0.0 when details exist"
assert _f["police_not_filed"] == 1.0, "(3) 'Not Filed' means a report was expected and skipped"
assert build_features({**_probe, "Police Report Number": "Not Required"},
                      1.5)["police_not_filed"] == 0.0, \
    "(3) 'Not Required' is normal for small claims — it must NOT fire"
assert build_features({**_probe, "Police Report Number": "PR-2026-1234"},
                      1.5)["police_not_filed"] == 0.0, "(3) a real report number must NOT fire"
assert _f["photo_severity"] == 1.5, "(4) should be the score passed into the function"
print("✅ Exercise 3 looks right.")


In [ ]:
def build_claim_record(claim_id):
    """Assemble everything known about one claim: fields, both readings, and features."""
    row = labels_df[labels_df.claim_id == claim_id].iloc[0]
    fields = parsed[claim_id]

    image_path = os.path.join(DATA_DIR, "images", row["image_file"])
    image_vec = embed_image(Image.open(image_path).convert("RGB"))
    photo_level, photo_score, _ = read_severity(image_vec)

    # The same grading applied to the form's own wording, so the two readings
    # are directly comparable rather than one being a hand-written word list.
    claimed_text = fields.get("Damage Claimed", "") or ""
    text_level, text_score, _ = (read_severity(embed_text(claimed_text))
                                 if claimed_text else ("minor", 0.0, None))

    return {
        "claim_id": claim_id,
        "label": row["label"],
        "image_path": image_path,
        "fields": fields,
        "photo_level": photo_level,
        "photo_severity": photo_score,
        "text_level": text_level,
        "text_severity": text_score,
        "cost": money(fields.get("Estimated Repair Cost")) or 0.0,
        "cover": money(fields.get("Sum Insured")) or 1.0,
        "features": build_features(fields, photo_score),
    }


records = {cid: build_claim_record(cid) for cid in ok_ids}
ids = list(records)
y = [records[c]["label"] for c in ids]

# This is literally the table being searched — one row per claim, one column per feature.
feature_table = pd.DataFrame(
    [[records[c]["features"][k] for k in FEATURE_NAMES] for c in ids],
    index=[f"{c} ({records[c]['label'][:1]})" for c in ids], columns=FEATURE_NAMES,
)
print(f"The index: {feature_table.shape[0]} claims x {feature_table.shape[1]} features\n")
feature_table.round(2)


### How those numbers get prepared

The raw table cannot be compared with cosine similarity as it stands, because the columns are on
wildly different scales — `speed` runs 0–65, `log_cost` sits around 3–4.5, and the indicators are 0
or 1. Cosine similarity would be dominated by `speed` for no better reason than its numbers being
bigger.

Two steps fix it, and both matter:

**1 · Standardise each column.** Subtract the mean, divide by the standard deviation, so every
feature is measured in "how unusual is this claim on this attribute" rather than in mph or dollars.
After this, a claim two standard deviations above average on cost counts exactly as much as one two
standard deviations above average on speed.

$$x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

**2 · Normalise each row to length 1.** Now cosine similarity is a plain dot product, and it compares
the *pattern* across features rather than the overall magnitude of a claim.

$$\hat{x}_i = \frac{x'_i}{\lVert x'_i \rVert}$$

One subtlety worth naming: $\mu_j$ and $\sigma_j$ are computed from the **index**, and a new claim is
projected into that same scale. The stored claims define the coordinate system — exactly as they
would in a real vector store. Re-fitting the scale for every incoming query would let one new claim
silently redefine the whole database.


In [ ]:
def scale_features(matrix, reference=None):
    """Standardise each column, then normalise each row to unit length.

    Args:
        matrix: (n_claims, n_features) array to scale.
        reference: Array whose column means and standard deviations define the
            scale. Defaults to `matrix` itself; when scaling a query we pass the
            index, so the stored claims define the coordinate system.

    Returns:
        Scaled, row-normalised array of the same shape as `matrix`.
    """
    ref = matrix if reference is None else reference
    out = (matrix - ref.mean(0)) / (ref.std(0) + 1e-9)
    return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-9)


def feature_matrix(claim_ids):
    """Stack the feature vectors for the given claims, in FEATURE_NAMES order."""
    return np.array([[records[c]["features"][k] for k in FEATURE_NAMES] for c in claim_ids])


F_raw = feature_matrix(ids)
F = scale_features(F_raw)
print("before scaling, the columns are not comparable:")
for j, name in enumerate(FEATURE_NAMES):
    print(f"    {name:22s} range {F_raw[:, j].min():6.2f} to {F_raw[:, j].max():6.2f}")
print(f"\nafter scaling: every column has mean 0 and sd 1, "
      f"and every row has length {np.linalg.norm(F[0]):.2f}")


In [ ]:
#@title 📊 Visualization: the index, one row per claim (double-click to view the code) { display-mode: "form" }
order = sorted(range(len(ids)), key=lambda i: (y[i] != "fraud", ids[i]))
fig, ax = plt.subplots(figsize=(9, 0.42 * len(ids) + 1.6))
im = ax.imshow(F[order], cmap="RdBu_r", vmin=-0.6, vmax=0.6, aspect="auto")

ax.set_xticks(range(len(FEATURE_NAMES)))
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(ids)))
ax.set_yticklabels([ids[i] for i in order], fontsize=8)
for tick, i in zip(ax.get_yticklabels(), order):
    tick.set_color(LABEL_COLOR[y[i]])

n_fraud = sum(1 for lab in y if lab == "fraud")
ax.axhline(n_fraud - 0.5, color="black", lw=1.4)
ax.set_title("What the retriever actually searches\n"
             "(standardised; red = above average, blue = below. Fraud above the line)",
             fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
plt.tight_layout()
plt.show()


### Before you build a retriever, check the representation can work

Before building any of it, there is a five-minute test worth running — and it generalises to any
retrieval system you are ever asked to evaluate.

For retrieval-by-similarity to help, claims **sharing an outcome must be more similar to each other
than to claims with the opposite outcome.** If that is not true of your representation, no amount of
tuning the search will save you — the answer simply is not encoded in the numbers you indexed.

So measure it: average similarity between same-outcome pairs, minus average similarity between
opposite-outcome pairs. Call it the **separation gap**. A gap near zero means the representation
knows nothing about your question.


In [ ]:
def separation_gap(matrix, labels):
    """Mean within-label similarity minus mean across-label similarity.

    Args:
        matrix: Square similarity matrix.
        labels: Sequence of labels aligned with the matrix rows.

    Returns:
        (within_mean, across_mean, gap).
    """
    n = len(labels)
    off_diagonal = ~np.eye(n, dtype=bool)
    same = np.array([[labels[i] == labels[j] for j in range(n)] for i in range(n)])
    within = matrix[off_diagonal & same].mean()
    across = matrix[off_diagonal & ~same].mean()
    return within, across, within - across


# Representation A — glue the CLIP vectors together (the tempting approach)
clip_matrix = np.array([
    np.concatenate([
        embed_image(Image.open(records[c]["image_path"]).convert("RGB")),
        embed_text(f"car damage: {records[c]['fields'].get('Damage Claimed', '')}"),
    ]) / np.sqrt(2) for c in ids
])
S_clip = clip_matrix @ clip_matrix.T

# Representation B — the eleven structured attributes, scaled as above
S_struct = F @ F.T

for name, S in [("CLIP image+text embeddings", S_clip), ("structured attributes", S_struct)]:
    w, a, gap = separation_gap(S, y)
    verdict = "✅ usable" if gap > 0.15 else "❌ no signal — do not build on this"
    print(f"{name:30s} within {w:+.3f} | across {a:+.3f} | gap {gap:+.3f}   {verdict}")


In [ ]:
#@title 📊 Visualization: does the representation separate the outcomes? (double-click to view the code) { display-mode: "form" }
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
order = sorted(range(len(ids)), key=lambda i: y[i])

for ax, (name, S) in zip(axes, [("CLIP image+text embeddings", S_clip),
                                ("Structured attributes", S_struct)]):
    M = S[np.ix_(order, order)]
    im = ax.imshow(M, cmap="RdYlGn", vmin=-1, vmax=1)
    ticks = [f"{ids[i]} ({y[i][:1]})" for i in order]
    ax.set_xticks(range(len(ticks))); ax.set_yticks(range(len(ticks)))
    ax.set_xticklabels(ticks, rotation=90, fontsize=6)
    ax.set_yticklabels(ticks, fontsize=6)
    _, _, gap = separation_gap(S, y)
    ax.set_title(f"{name}\nseparation gap {gap:+.3f}", fontsize=10)
    boundary = sum(1 for lab in y if lab == "fraud") - 0.5
    ax.axhline(boundary, color="black", lw=1.4)
    ax.axvline(boundary, color="black", lw=1.4)

fig.colorbar(im, ax=axes, fraction=0.02)
plt.suptitle("Same claims, same maths — only the representation changed",
             fontsize=12, y=1.12)
plt.show()


**Reading the result.** In the left panel almost every square is the same shade: every claim is
similar to every claim, and the black lines separating fraud from non-fraud pass through nothing. In
the right panel the block structure is visible — the two groups genuinely sit apart.

That separation is what makes retrieval usable, so now we can turn it into a prediction. The rule is
the simplest one there is: **retrieve the `k` most similar past claims and take a majority vote.**
Three neighbours, two of them fraudulent, and the claim is called fraud. Nothing is learned, nothing
is fitted — the verdict is just what the nearest precedents did, which is also why it can be shown to
an adjuster as a list of case numbers rather than a score.

This is the lesson of Part 4, and it is the one to carry into a vendor meeting: **RAG quality is
decided by what you index, not by which retrieval algorithm or vector database you buy.** Same
k-nearest-neighbours, same majority vote, same cosine similarity, same claims. Only the
representation changed.


In [ ]:
def retrieve_similar(query_features, index_ids, top_k=3):
    """Find the claims whose risk structure most resembles the query.

    Args:
        query_features: Feature dict for the claim being assessed.
        index_ids: Claim ids to search over (the query must not be among them).
        top_k: How many neighbours to return.

    Returns:
        List of (claim_id, label, similarity), most similar first.
    """
    index_matrix = feature_matrix(index_ids)
    query_row = np.array([[query_features[k] for k in FEATURE_NAMES]])

    # Both are scaled against the INDEX, so the stored claims define the
    # coordinate system and an incoming claim cannot rescale the database.
    index_scaled = scale_features(index_matrix)
    query_scaled = scale_features(query_row, reference=index_matrix)[0]

    sims = index_scaled @ query_scaled
    ranked = sorted(zip(index_ids, sims), key=lambda t: -t[1])[:top_k]
    return [(cid, records[cid]["label"], float(s)) for cid, s in ranked]


def evaluate_retrieval(top_k=3):
    """Leave-one-out: predict each claim from its neighbours among all the others."""
    correct, rows = 0, []
    for cid in ids:
        others = [c for c in ids if c != cid]
        neighbours = retrieve_similar(records[cid]["features"], others, top_k=top_k)
        votes = [lab for _, lab, _ in neighbours]
        pred = "fraud" if votes.count("fraud") >= votes.count("non_fraud") else "non_fraud"
        correct += pred == records[cid]["label"]
        rows.append({"claim": cid, "true": records[cid]["label"], "predicted": pred,
                     "neighbours": ", ".join(f"{c}({l[:1]})" for c, l, _ in neighbours)})
    return correct / len(ids), pd.DataFrame(rows)


baseline = labels_df["label"].value_counts(normalize=True).max()
print(f"Baseline (always guess the majority): {baseline:.0%}\n")
for k in (1, 3, 5):
    acc, _ = evaluate_retrieval(top_k=k)
    print(f"  retrieval with top_k={k}: {acc:.0%}")

acc3, detail = evaluate_retrieval(top_k=3)
print()
detail


**Reading the result.** Retrieval over structured attributes lands well above the baseline, and —
more importantly for a triage desk — the neighbours it returns are ones a human would accept. Claims
retrieve other claims with the same shape of story, not merely other photographs of dented metal.

That is what makes this *retrieval-augmented*: the system can hand an adjuster three specific past
files and say "this looks like these, and here is how they turned out."


In [ ]:
#@title 📊 Visualization: pick a claim and see what retrieval pulls back { display-mode: "form" }
def show_retrieval(query_id, top_k=3):
    """Show one claim beside the past claims retrieved for it.

    Args:
        query_id: The claim to use as the query.
        top_k: How many neighbours to display.
    """
    index_ids = [c for c in ids if c != query_id]
    neighbours = retrieve_similar(records[query_id]["features"], index_ids, top_k=top_k)
    q = records[query_id]

    def caption(rec):
        """One line of the risk structure behind the match."""
        return (f"${rec['cost']:,.0f} · {rec['features']['cost_ratio']:.0%} of cover\n"
                f"photo {rec['photo_level']} · form {rec['text_level']}")

    fig, axes = plt.subplots(1, top_k + 1, figsize=(2.7 * (top_k + 1), 3.6))
    axes[0].imshow(Image.open(q["image_path"]).convert("RGB"))
    axes[0].set_title(f"QUERY — {query_id}\n({q['label']})", fontsize=9, color=PURPLE2)
    axes[0].set_xlabel(caption(q), fontsize=7.5, color="#555")
    for side in axes[0].spines.values():
        side.set_visible(True); side.set_color(PURPLE2); side.set_linewidth(3)

    for ax, (cid, label, sim) in zip(axes[1:], neighbours):
        rec = records[cid]
        ax.imshow(Image.open(rec["image_path"]).convert("RGB"))
        ax.set_title(f"{cid} ({label})\nsimilarity {sim:.2f}", fontsize=9,
                     color=LABEL_COLOR[label])
        ax.set_xlabel(caption(rec), fontsize=7.5, color="#555")
        for side in ax.spines.values():
            side.set_visible(True); side.set_color(LABEL_COLOR[label]); side.set_linewidth(3)

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])

    votes = [lab for _, lab, _ in neighbours]
    verdict, share = precedent_vote(neighbours) if "precedent_vote" in globals() else (
        ("fraud" if votes.count("fraud") > len(votes) / 2 else "non_fraud"),
        votes.count("fraud") / len(votes))
    plt.suptitle(f"Retrieved by risk structure — majority vote: {verdict.upper()} "
                 f"({votes.count('fraud')} of {len(votes)} neighbours fraudulent)",
                 fontsize=11)
    plt.tight_layout()
    plt.show()


# Interactive where widgets are available, static otherwise.
DEFAULT_QUERY = "claim_11" if "claim_11" in records else ids[0]
try:
    from ipywidgets import interact, Dropdown, IntSlider
    interact(show_retrieval,
             query_id=Dropdown(options=ids, value=DEFAULT_QUERY, description="claim:"),
             top_k=IntSlider(min=1, max=5, step=1, value=3, description="neighbours:"))
except Exception:
    print("(ipywidgets unavailable — showing one claim; edit DEFAULT_QUERY to change it)")
    show_retrieval(DEFAULT_QUERY)


---

## Part 5 — A new claim lands on your desk

Everything so far has been machinery: read the form, read the photograph, find precedent. Now put it
to work on the thing the job is actually about — **one claim arrives, and you have to decide.**

Retrieval tells you *this resembles those*. It does not tell you *why*, and a referral needs a reason
an adjuster can read, argue with, and overrule. So alongside precedent we run four explicit checks —
the ones an experienced handler applies by instinct. They fall into **three families**, each asking a
different question and each failing independently:

| family | the question it asks | needs the photo? |
|---|---|---|
| **Corroboration** | Can anyone independently confirm this story? | no |
| **Proportionality** | Is the money proportionate to the policy? | no |
| **Consistency** | Do the photo and the form describe the same event? | **yes** |

**Corroboration — is there a witness?**

- `counterparty_unverifiable` — another vehicle is blamed, but no contact details were taken. There is
  nobody who can contradict the account.
- `police_report_not_filed` — a report was expected and never filed. (*"Not Required"* is normal for
  minor damage; anything else means somebody chose not to involve the police.)

**Proportionality — is the ask reasonable?**

- `large_claim_vs_cover` — the repair estimate exceeds a third of the total sum insured. Pure
  arithmetic on two numbers from the form, and no photograph is involved.

**Consistency — do the two accounts agree?**

- `photo_worse_than_described` — this is the multimodal one, and it is a different kind of question
  entirely. The other three interrogate the form against itself or against the policy. This one sets
  the form against *independent evidence*. We graded the photograph in Part 3 and we grade the form's
  own `Damage Claimed` wording on the **same scale with the same three captions**. If the picture says
  *severe* and the paperwork says *minor*, the two accounts of one event disagree — and no amount of
  reading the document more carefully would ever reveal that.

No keyword list, no hand-tuned vocabulary of suspicious words. Both readings come out of one shared
space, which is what the lecture's multimodal framing actually buys.

We define the checks, see whether they discriminate on the claims we already have, ask whether the
photograph earns its place, then bring retrieval back as a second, independent opinion — and only
then run the whole thing end to end on a claim the system has never seen.


### Exercise 4. The check only the photograph can make

Three of the four flags read the form against itself or against the policy. This one is different: it
sets the form against **independent evidence**.

Both readings are already on the record and both came from the *same three captions*, so they are
directly comparable — `record["photo_level"]` is what the picture shows, `record["text_level"]` is
what the policyholder wrote. When the picture says *severe* and the paperwork says *minor*, the two
accounts of one event do not match.

This is one line, and it is the line the notebook has been building towards.

**Task.** Complete the condition for `photo_worse_than_described`.


#### Your turn


In [ ]:
def red_flags(record):
    """Apply the corroboration, proportionality and consistency checks to one claim.

    Args:
        record: A claim record from build_claim_record.

    Returns:
        Dict of flag name to (fired: bool, human-readable explanation).
    """
    f = record["fields"]
    police = str(f.get("Police Report Number", "")).strip().lower()
    tp_involved = str(f.get("Third Party Involved", "")).strip().lower()
    tp_info = str(f.get("Third Party Information Available", "")).strip().lower()
    ratio = record["features"]["cost_ratio"]

    flags = {}

    # ── Corroboration ──
    flags["counterparty_unverifiable"] = (
        tp_involved in ("yes", "unknown") and tp_info == "no",
        "Another vehicle is blamed but no contact details were obtained, "
        "so nobody can confirm the account.",
    )
    # Read this as "is there a report number?" rather than matching the exact
    # words. Free-text categorical fields drift -- this dataset alone contains
    # "Not Filed", "Not filed" and "Not Field" -- so an == comparison silently
    # misses claims. A reference number is either present or it is not.
    has_report = police.startswith("pr-")
    not_required = police == "not required"
    flags["police_report_not_filed"] = (
        not has_report and not not_required,
        "No police report number is recorded. ('Not Required' is normal for minor "
        "damage; anything else means one was expected and never filed.)",
    )

    # ── Proportionality: is the ask reasonable against the policy? ──
    flags["large_claim_vs_cover"] = (
        ratio > 0.35,
        f"The repair estimate is {ratio:.0%} of the total sum insured.",
    )
    # ── Consistency: do the photograph and the form describe the same event? ──
    # Both readings come from the SAME three captions, so "severe" from the photo
    # and "minor" from the form are directly comparable.
    flags["photo_worse_than_described"] = (
        # 🎯 YOUR TURN — Exercise 4: fire when the photograph is worse than the form admits.
        #    Hint: True when record["photo_level"] is "severe" AND record["text_level"]
        #    is "minor". One `and`, two string comparisons.
        ...,
        f"The form describes {record['text_level']} damage "
        f"('{(f.get('Damage Claimed') or '')[:46]}') but the photograph reads as "
        f"{record['photo_level']} — the two accounts do not match.",
    )
    return flags



In [ ]:
# ✅ Self-check — run this straight after your answer.
def _probe(photo_level, text_level):
    """A minimal fake record, just enough for the flag under test."""
    return {"fields": {"Damage Claimed": "minor paint scratches"},
            "features": {"cost_ratio": 0.10},
            "photo_level": photo_level, "text_level": text_level}


assert red_flags(_probe("severe", "minor"))["photo_worse_than_described"][0], \
    "a severe photo against a minor form should fire"
assert not red_flags(_probe("minor", "minor"))["photo_worse_than_described"][0], \
    "when both read 'minor' the accounts agree — it must not fire"
assert not red_flags(_probe("severe", "severe"))["photo_worse_than_described"][0], \
    "when both read 'severe' the accounts agree — it must not fire"
assert not red_flags(_probe("moderate", "minor"))["photo_worse_than_described"][0], \
    "only a 'severe' photo counts as contradicting a 'minor' form"
print("✅ Exercise 4 looks right.")


In [ ]:
FLAG_FAMILY = {
    "counterparty_unverifiable": "corroboration",
    "police_report_not_filed": "corroboration",
    "large_claim_vs_cover": "proportionality",
    "photo_worse_than_described": "consistency",      # the only one needing the photo
}

flag_table = pd.DataFrame([
    {"claim": cid, "label": records[cid]["label"],
     "photo": records[cid]["photo_level"], "form": records[cid]["text_level"],
     **{name: fired for name, (fired, _) in red_flags(records[cid]).items()}}
    for cid in ids
])
flag_table["n_flags"] = flag_table[list(FLAG_FAMILY)].sum(axis=1)
flag_table


In [ ]:
# How discriminating is each check on its own?
print(f"{'flag':30s} {'family':16s} {'fires on fraud':>15s} {'on non-fraud':>14s}")
n_fraud = (flag_table.label == "fraud").sum()
n_clean = (flag_table.label == "non_fraud").sum()
for name, family in FLAG_FAMILY.items():
    fr = int(flag_table[flag_table.label == "fraud"][name].sum())
    nf = int(flag_table[flag_table.label == "non_fraud"][name].sum())
    note = "  ← no false positives" if nf == 0 else "  ← FALSE POSITIVES"
    print(f"{name:30s} {family:16s} {fr:>10}/{n_fraud} {nf:>10}/{n_clean}{note}")

flag_table["predicted"] = np.where(flag_table["n_flags"] >= 1, "fraud", "non_fraud")
accuracy = (flag_table["predicted"] == flag_table["label"]).mean()
print(f"\nRule 'any flag fires → refer':  {accuracy:.0%}   (baseline {baseline:.0%})")


### Does the photograph actually earn its keep?

A multimodal pipeline should be made to justify itself. Ours uses the photo in three places, so the
honest question is: **if we deleted the image entirely, would any fraud slip through?**

Run the ablation rather than assuming — the habit matters more than the answer.


In [ ]:
doc_only = [n for n, fam in FLAG_FAMILY.items() if n != "photo_worse_than_described"]
photo_only = ["photo_worse_than_described"]

for name, cols in [("document flags only", doc_only),
                   ("photo flag only", photo_only),
                   ("both", list(FLAG_FAMILY))]:
    pred = np.where(flag_table[cols].any(axis=1), "fraud", "non_fraud")
    print(f"  {name:22s} {(pred == flag_table['label']).mean():.0%}")

caught_only_by_photo = flag_table[
    (flag_table.label == "fraud")
    & flag_table["photo_worse_than_described"]
    & ~flag_table[doc_only].any(axis=1)
]
print(f"\nFraud claims caught ONLY by the photograph: {len(caught_only_by_photo)}"
      f" {list(caught_only_by_photo['claim']) if len(caught_only_by_photo) else ''}")


**Reading the result.** Deleting the image is not free. The document checks alone drop below the
combined rule, and **two fraudulent claims are caught by nothing but the photograph.**

Look at what those two have in common. `claim_06` and `claim_11` have *immaculate paperwork*: a police
report was filed, no third party is blamed, and the repair estimate is a modest 14% and 21% of cover.
Every corroboration and proportionality check on the document side passes. On the form they are
unremarkable claims.

Then you look at the photographs. `claim_06`'s form describes *"front fender dent, minor paint
scratches"* from a shopping trolley in a windstorm; the picture shows a car with its front end
destroyed and the A-pillar torn open. `claim_11` claims a *"minor front fender dent"* at 2 mph, and
the photograph shows the quarter panel ripped away with the inner structure exposed.

**A document-only pipeline waves both of these straight through to payment.** The only thing that
contradicts the story is the image — and the only reason the two can be compared at all is that the
same three captions grade both.

That is the case for multimodality stated properly: not "more inputs are better", but *this specific
question could not be asked of the document alone.*

### A second opinion: what does precedent say?

The flags are hand-written. Somebody had to know that an unreachable third party is suspicious — and
a rule only ever catches what its author thought of.

Retrieval gives us a completely different kind of judgement, built from no rules at all. We already
have a way to find the claims most similar in risk structure (Part 4). So: **pull the three nearest
past claims and let them vote.** If most of them turned out fraudulent, that is evidence, and it
required nobody to write down what fraud looks like.

Two independent assessors, then — one reasoning from rules, one from precedent. The interesting
question is not just which is more accurate, but **what happens when they disagree.**


### Exercise 5. Let precedent vote

`retrieve_similar` hands back the `k` most similar past claims, each with the outcome it eventually
had. Turning that into a verdict is a plain count — no model, no training, no threshold to tune.

Deliberately a *count* rather than a similarity-weighted average, because the result has to be
sayable out loud: **"two of the three most similar past claims were fraudulent."**

**Task.** Complete `precedent_vote` so it returns the verdict and the fraudulent share.


#### Your turn


In [ ]:
# 🎯 YOUR TURN — Exercise 5: let the retrieved neighbours vote.
#
# 💭 Think first: with three neighbours a majority is two. What would happen with
#    four neighbours split two-two — and which way should a triage system lean when
#    the evidence is tied?

def precedent_vote(precedent):
    """Let the retrieved neighbours vote on the verdict.

    Args:
        precedent: List of (claim_id, label, similarity) from retrieve_similar.

    Returns:
        (verdict, fraud_share) where verdict is one of LABELS.
    """
    labels = [label for _, label, _ in precedent]   # e.g. ["fraud", "fraud", "non_fraud"]
    if not labels:
        return "non_fraud", 0.0

    # 🎯 Implement `fraud_share`: what fraction of the neighbours were fraudulent?
    #    Hint: `labels.count("fraud")` counts them; divide by `len(labels)`.
    fraud_share = ...

    # 🎯 Implement `verdict`: "fraud" when more than half voted fraud, otherwise "non_fraud".
    #    Hint: a one-line if/else expression — "fraud" if <condition> else "non_fraud"
    verdict = ...

    return verdict, fraud_share


In [ ]:
# ✅ Self-check — run this straight after your answer.
_two_of_three = [("a", "fraud", 0.9), ("b", "fraud", 0.8), ("c", "non_fraud", 0.7)]
_one_of_three = [("a", "fraud", 0.9), ("b", "non_fraud", 0.8), ("c", "non_fraud", 0.7)]

_v, _s = precedent_vote(_two_of_three)
assert _v == "fraud", f"two of three fraudulent is a fraud majority, got {_v!r}"
assert abs(_s - 2 / 3) < 1e-9, f"the share should be 2/3, got {_s:.3f}"
assert precedent_vote(_one_of_three)[0] == "non_fraud", \
    "one of three fraudulent is not a majority"
assert precedent_vote([("a", "non_fraud", 0.5)])[0] == "non_fraud", \
    "a single honest neighbour should give non_fraud"
print("✅ Exercise 5 looks right.")


In [ ]:
# Compare the two assessors on every claim, each judged against all the others.
comparison = []
for cid in ids:
    others = [c for c in ids if c != cid]
    neighbours = retrieve_similar(records[cid]["features"], others, top_k=3)
    precedent_verdict, share = precedent_vote(neighbours)
    rules_verdict = ("fraud" if any(hit for hit, _ in red_flags(records[cid]).values())
                     else "non_fraud")
    comparison.append({
        "claim": cid, "true_label": records[cid]["label"],
        "rules_say": rules_verdict, "precedent_says": precedent_verdict,
        "fraud_share": share, "agree": rules_verdict == precedent_verdict,
    })

comparison_df = pd.DataFrame(comparison)
rules_acc = (comparison_df.rules_say == comparison_df.true_label).mean()
prec_acc = (comparison_df.precedent_says == comparison_df.true_label).mean()

print(f"rules alone      {rules_acc:.0%}")
print(f"precedent alone  {prec_acc:.0%}")
print(f"baseline         {baseline:.0%}\n")

agreed = comparison_df[comparison_df.agree]
disagreed = comparison_df[~comparison_df.agree]
print(f"They agree on {len(agreed)}/{len(comparison_df)} claims, "
      f"and where they agree they are right {(agreed.rules_say == agreed.true_label).mean():.0%} "
      f"of the time.\n")
print(f"They disagree on {len(disagreed)}:")
print(disagreed[["claim", "true_label", "rules_say", "precedent_says"]].to_string(index=False))
print(f"\nOn those, rules were right {(disagreed.rules_say == disagreed.true_label).sum()} time(s), "
      f"precedent {(disagreed.precedent_says == disagreed.true_label).sum()}.")


**Reading the result.** Precedent alone is a strong signal — far above the baseline, built from no
rules whatsoever, and close to the hand-written checks. But combining the two by "flag it if either
says fraud" would make things *worse*, not better: it inherits precedent's false alarms without
catching anything the rules missed. On the claims where the two disagree, the rules are right.

That is not surprising, and it is worth saying why. The rules encode what an experienced claims
handler knows. Retrieval has to *infer* the same thing from seventeen labelled examples, which is
nowhere near enough for "similar risk structure" to reliably mean "similar outcome". With a few
thousand historical claims the balance would likely shift.

**So do not use precedent as a vote to add on. Use it as a check on your confidence.**

Where the two assessors agree, they are right every time — and that is the majority of claims, which
can be routed automatically. Where they disagree, something is genuinely ambiguous, and *those* are
the files worth a human's attention. The value is not a better score; it is knowing **which decisions
you are entitled to make automatically.**


In [ ]:
# Route on agreement rather than on either verdict alone.
auto = comparison_df[comparison_df.agree]
escalate = comparison_df[~comparison_df.agree]

print(f"Routed automatically (both assessors agree):  {len(auto):2d}/{len(comparison_df)} claims"
      f"   accuracy {(auto.rules_say == auto.true_label).mean():.0%}")
print(f"Escalated to a human (they disagree):         {len(escalate):2d}/{len(comparison_df)} claims")
print(f"\nSo {len(auto) / len(comparison_df):.0%} of the queue clears without a human ever seeing it,")
print(f"and the {len(escalate)} genuinely ambiguous claims get the attention they deserve.")
print("\nThat is a triage system doing its actual job: not replacing the adjuster,")
print("but deciding which claims need one.")


### Now the real thing: a claim the system has never seen

The checks above were measured on claims already in the book. That is how you tune a system, not how
you trust one. So we hold a claim out of the index entirely and hand the system its photograph, its
PDF, and the question a triage lead would actually ask.


In [ ]:
#@title 🏗️ Visualization: the assessment pipeline (double-click to view the code) { display-mode: "form" }
def _pipeline_diagram():
    """Render the assessment flow as a card diagram."""
    uid = uuid.uuid4().hex[:8]
    steps = [
        ("1", "📄", "Read the document", "pdfplumber → fields, then validate the parse"),
        ("2", "🖼️", "Look at the photo", "CLIP zero-shot → damage severity"),
        ("3", "🤝", "Corroboration", "Can anyone confirm this story?"),
        ("4", "⚖️", "Proportionality", "Is the money proportionate to the policy?"),
        ("5", "🔍", "Consistency", "Do photo and form describe the same event?"),
        ("6", "🔎", "Precedent", "A second opinion from the nearest past claims"),
        ("7", "📝", "Decide + explain", "A verdict, and one sentence of why"),
    ]
    cards = "".join(f"""
      <div style="flex:1 1 210px;min-width:210px;background:#fff;border:1px solid #e6e8ee;
                  border-radius:14px;padding:14px 16px;">
        <div style="display:flex;align-items:center;gap:10px;margin-bottom:6px;">
          <span style="width:26px;height:26px;border-radius:50%;flex:none;
                background:linear-gradient(135deg,{PURPLE},{PURPLE2});color:#fff;font-size:13px;
                text-align:center;line-height:26px;font-weight:700;">{num}</span>
          <span style="font-size:17px;">{icon}</span>
          <span style="font-weight:650;color:#2d2f45;font-size:13.5px;">{title}</span>
        </div>
        <div style="color:#6a6d85;font-size:12px;line-height:1.4;">{sub}</div>
      </div>""" for num, icon, title, sub in steps)
    return HTML(f"""
    <div id="pl{uid}" style="font-family:system-ui,Segoe UI,Roboto,sans-serif;border-radius:18px;
         border:1px solid #ecebff;padding:22px;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);">
      <div style="font-weight:700;color:#2d2f45;margin-bottom:16px;font-size:15px;">
        What happens when a claim lands on the desk</div>
      <div style="display:flex;flex-wrap:wrap;gap:12px;">{cards}</div>
    </div>""")

display(_pipeline_diagram())


In [ ]:
def assess_claim(claim_id, index_ids, question, top_k=3):
    """Run the full triage on one claim: checks, precedent, and a routed decision.

    Args:
        claim_id: The claim to assess.
        index_ids: Historical claims to retrieve from; must exclude claim_id.
        question: The triage question being asked, in plain language.
        top_k: How many precedents to retrieve.

    Returns:
        A dict holding the decision, the fired flags, the retrieved precedent,
        and the evidence bundle a language model would be given to write this up.
    """
    record = records[claim_id]

    flags = red_flags(record)                                   # the rules assessor
    fired = {name: why for name, (hit, why) in flags.items() if hit}
    rules_verdict = "fraud" if fired else "non_fraud"

    precedent = retrieve_similar(record["features"], index_ids, top_k=top_k)   # the precedent assessor
    precedent_verdict, fraud_share = precedent_vote(precedent)

    # Route on agreement: act only where both assessors reach the same verdict.
    if rules_verdict != precedent_verdict:
        decision = "SECOND OPINION NEEDED"
    elif rules_verdict == "fraud":
        decision = "REFER TO INVESTIGATOR"
    else:
        decision = "FAST-TRACK"

    return {
        "claim_id": claim_id,
        "question": question,
        "decision": decision,
        "photo_reads": record["photo_level"],
        "claimed_damage": record["fields"].get("Damage Claimed", ""),
        "cost": record["cost"],
        "cover": record["cover"],
        "fired_flags": fired,
        "precedent": precedent,
        "rules_verdict": rules_verdict,
        "precedent_verdict": precedent_verdict,
        "assessors_agree": rules_verdict == precedent_verdict,
        "precedent_fraud_share": fraud_share,
        "true_label": record["label"],
    }


# Hold one claim out of the index entirely — the system has never seen it.
NEW_CLAIM = "claim_06" if "claim_06" in records else ids[-1]
INDEX = [c for c in ids if c != NEW_CLAIM]
QUESTION = "Should this claim be fast-tracked for payment, or referred to an investigator?"

result = assess_claim(NEW_CLAIM, INDEX, QUESTION)
print(f"Index holds {len(INDEX)} historical claims. Assessing {NEW_CLAIM}, unseen.\n")
print(f"Q: {result['question']}")
print(f"   rules say     : {result['rules_verdict']}  ({len(result['fired_flags'])} flag(s) raised)")
print(f"   precedent says: {result['precedent_verdict']}  "
      f"({result['precedent_fraud_share']:.0%} of neighbours were fraudulent)")
print(f"A: {result['decision']}")


In [ ]:
#@title 📊 Visualization: the claim as the system sees it (double-click to view the code) { display-mode: "form" }
rec = records[NEW_CLAIM]
fig = plt.figure(figsize=(11, 3.6))

ax_img = fig.add_subplot(1, 3, 1)
ax_img.imshow(Image.open(rec["image_path"]).convert("RGB"))
ax_img.set_xticks([]); ax_img.set_yticks([])
ax_img.set_title(f"{NEW_CLAIM}: what was submitted", fontsize=10)

ax_txt = fig.add_subplot(1, 3, 2)
ax_txt.axis("off")
lines = [
    f"Damage claimed:  {(rec['fields'].get('Damage Claimed') or '')[:34]}",
    f"Repair estimate: ${rec['cost']:,.0f}",
    f"Sum insured:     ${rec['cover']:,.0f}",
    f"Claim / cover:   {rec['features']['cost_ratio']:.0%}",
    f"Police report:   {rec['fields'].get('Police Report Number', '?')}",
    f"Third party:     {rec['fields'].get('Third Party Involved', '?')}",
    f"Vehicle speed:   {rec['fields'].get('Vehicle Speed', '?')}",
]
ax_txt.text(0, 0.95, "WHAT THE FORM SAYS", fontsize=9.5, fontweight="bold", va="top")
ax_txt.text(0, 0.80, "\n".join(lines), fontsize=8.5, va="top", family="monospace")

ax_bar = fig.add_subplot(1, 3, 3)
_, _, pr = read_severity(embed_image(Image.open(rec["image_path"]).convert("RGB")))
ax_bar.barh(SEVERITY_LEVELS, pr, color=[GREEN, AMBER, RED])
ax_bar.set_xlim(0, 1)
ax_bar.set_xlabel("zero-shot probability", fontsize=8)
ax_bar.set_title("WHAT THE PHOTO SHOWS", fontsize=9.5, fontweight="bold")
ax_bar.tick_params(labelsize=8)

plt.tight_layout()
plt.show()


In [ ]:
#@title 📊 Visualization: the precedent behind this decision { display-mode: "form" }
query_record = records[NEW_CLAIM]
retrieved = result["precedent"]


def _claim_panel(ax, record, title, colour):
    """Draw one claim: its photograph, plus the risk structure underneath."""
    ax.imshow(Image.open(record["image_path"]).convert("RGB"))
    ax.set_title(title, fontsize=9, color=colour)
    ax.set_xlabel(f"${record['cost']:,.0f} · {record['features']['cost_ratio']:.0%} of cover\n"
                  f"photo {record['photo_level']} · form {record['text_level']}",
                  fontsize=7.5, color="#555")
    ax.set_xticks([]); ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(True); side.set_color(colour); side.set_linewidth(3)


fig, axes = plt.subplots(1, len(retrieved) + 1, figsize=(2.75 * (len(retrieved) + 1), 3.9))
_claim_panel(axes[0], query_record,
             f"NEW CLAIM — {NEW_CLAIM}\n(held out of the index)", PURPLE2)

for ax, (cid, label, similarity) in zip(axes[1:], retrieved):
    _claim_panel(ax, records[cid], f"{cid} ({label})\nsimilarity {similarity:.2f}",
                 LABEL_COLOR[label])

fraud_count = sum(1 for _, label, _ in retrieved if label == "fraud")
plt.suptitle(
    f"What the index returned for {NEW_CLAIM} — majority vote: "
    f"{result['precedent_verdict'].upper()} "
    f"({fraud_count} of {len(retrieved)} neighbours fraudulent)", fontsize=11)
plt.tight_layout()
plt.show()


**Reading the result.** The photographs across that row have very little in common — different cars,
different damage, different angles. They were not retrieved for looking alike. They were retrieved
because their *risk structure* matches: comparable share of cover, comparable corroboration, a
comparable gap between what the photo shows and what the form claims.

That is what the decision cites. Not "the model scored 0.87", but three named files an adjuster can
pull, read, and disagree with.


In [ ]:
#@title 📝 Visualization: the triage decision (double-click to view the code) { display-mode: "form" }
def _decision_card(res):
    """Render the assessment as the panel an adjuster would see."""
    uid = uuid.uuid4().hex[:8]
    accent = (AMBER if not res["assessors_agree"]
              else RED if res["decision"].startswith("REFER") else GREEN)

    flags_html = "".join(f"""
      <div style="background:#fff;border:1px solid #e6e8ee;border-left:4px solid {AMBER};
                  border-radius:10px;padding:11px 14px;margin-bottom:8px;">
        <div style="font-weight:650;color:#2d2f45;font-size:12.5px;">🚩 {name.replace('_', ' ')}</div>
        <div style="color:#6a6d85;font-size:12px;margin-top:3px;line-height:1.4;">{why}</div>
      </div>""" for name, why in res["fired_flags"].items()) or """
      <div style="color:#6a6d85;font-size:12.5px;">No red flags raised.</div>"""

    rows = "".join(f"""
      <tr><td style="padding:4px 12px 4px 0;font-size:12.5px;color:#2d2f45;">{cid}</td>
          <td style="padding:4px 12px 4px 0;font-size:12.5px;color:{LABEL_COLOR[lab]};
                     font-weight:600;">{lab}</td>
          <td style="padding:4px 0;font-size:12.5px;color:#6a6d85;">similarity {s:.2f}</td></tr>"""
        for cid, lab, s in res["precedent"])

    return HTML(f"""
    <div id="dc{uid}" style="font-family:system-ui,Segoe UI,Roboto,sans-serif;border-radius:18px;
         border:1px solid #ecebff;padding:22px;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);">
      <div style="color:#6a6d85;font-size:12.5px;">{res['question']}</div>
      <div style="font-size:22px;font-weight:750;color:{accent};margin:6px 0 4px;">
        {res['decision']}</div>
      <div style="color:#6a6d85;font-size:12px;margin-bottom:6px;">
        claim {res['claim_id']} · photo reads <b>{res['photo_reads']}</b> ·
        ${res['cost']:,.0f} claimed against ${res['cover']:,.0f} cover</div>
      <div style="font-size:12.5px;margin-bottom:16px;">
        <span style="color:{LABEL_COLOR[res['rules_verdict']]};font-weight:600;">
          rules: {res['rules_verdict']}</span>
        <span style="color:#b9bccd;"> · </span>
        <span style="color:{LABEL_COLOR[res['precedent_verdict']]};font-weight:600;">
          precedent: {res['precedent_verdict']}</span>
        <span style="color:#6a6d85;"> — {'they agree' if res['assessors_agree'] else 'they disagree, so a human decides'}</span>
      </div>
      <div style="display:flex;flex-wrap:wrap;gap:16px;">
        <div style="flex:1 1 320px;min-width:300px;">
          <div style="font-weight:700;color:#2d2f45;font-size:13px;margin-bottom:8px;">
            Why — checks that fired</div>{flags_html}</div>
        <div style="flex:1 1 240px;min-width:230px;background:#fff;border:1px solid #e6e8ee;
                    border-radius:14px;padding:14px 16px;">
          <div style="font-weight:700;color:#2d2f45;font-size:13px;margin-bottom:8px;">
            Precedent retrieved</div>
          <table style="border-collapse:collapse;">{rows}</table>
          <div style="color:#6a6d85;font-size:11.5px;margin-top:10px;line-height:1.4;">
            {res['precedent_fraud_share']:.0%} of the most similar past claims were fraudulent.</div>
        </div>
      </div>
    </div>""")

display(_decision_card(result))
print(f"(Ground truth for {NEW_CLAIM}: {result['true_label']})")


**Reading the result.** The system did not produce a score. It produced a **decision with a
paper trail**: which checks fired and in plain words why, plus three named historical claims and how
they turned out. An adjuster can disagree with any line of it — and that is the point. A triage tool
that cannot be argued with is one that gets ignored the first time it is wrong.

Notice too that this is the shape of a *retrieval-augmented* system. Retrieval decided **what
evidence goes in front of the decision**, and the decision cites that evidence.


### The prompt this would hand to a language model

The final step in a full RAG system is **generation**: a model writes the case note. We stop one
step short and print the prompt instead, because the assembly *is* the lesson — everything the model
would say has to be grounded in the evidence bundle below, and if a fact is not in this block, a
generated answer that states it is a hallucination.


In [ ]:
def build_prompt(res):
    """Assemble the retrieved evidence into the prompt a case-note writer would receive."""
    flags = "\n".join(f"  - {n.replace('_', ' ')}: {w}" for n, w in res["fired_flags"].items())
    prec = "\n".join(f"  - {c}: outcome was {l} (similarity {s:.2f})"
                     for c, l, s in res["precedent"])
    return f"""You are assisting a motor-claims triage lead. Use ONLY the evidence below.
If the evidence does not support a statement, do not make it.

QUESTION
{res['question']}

CLAIM UNDER REVIEW: {res['claim_id']}
  Damage claimed by policyholder: {res['claimed_damage']}
  Repair estimate: ${res['cost']:,.0f} against ${res['cover']:,.0f} of cover
  Independent reading of the photograph: {res['photo_reads']} damage

AUTOMATED CHECKS THAT FIRED
{flags or '  (none)'}

TWO INDEPENDENT ASSESSORS
  - rule-based checks say: {res['rules_verdict']}
  - retrieved precedent says: {res['precedent_verdict']} ({res['precedent_fraud_share']:.0%} of the nearest past claims were fraudulent)
  - they {'agree' if res['assessors_agree'] else 'DISAGREE, so this needs a human'}

MOST SIMILAR HISTORICAL CLAIMS
{prec}

Write a two-sentence recommendation for the adjuster's file. Cite the specific
checks and past claims that justify it."""


print(build_prompt(result))


Everything in that prompt was **retrieved or computed** — nothing was invented. That constraint is
what separates a RAG system from a chatbot with opinions: the model's job is to phrase the evidence,
not to supply it.

Adding a real API call here is a small change, and deliberately left out: it needs a key, it costs
money, and it would not teach anything the prompt above does not already show.


---

## Summary

1. **Structure beats text.** The single most valuable step was turning a PDF of tables into named
   fields. Flattened to a paragraph, the form could not answer "is this repair estimate more than a
   third of the cover?" — parsed into fields, that is one division.

2. **Validate what your parser returns.** Document parsing fails *silently*: it does not raise, it
   returns something plausible-looking and carries on. Two checks cost three lines each — is the
   extraction empty, and did any value come back equal to a field label — and either would have
   caught a real defect in this dataset.

3. **What you index decides what you can find.** The identical retrieval algorithm went from useless
   to useful purely by changing the representation — from generic CLIP embeddings to the structured
   attributes the decision actually depends on. When retrieval underperforms, the question is not
   "which vector database?" but "what did we index, and does it encode the decision?"

4. **Check the representation before you build on it.** The separation gap — how much more similar
   same-outcome claims are than different-outcome ones — takes three lines and predicts in advance
   whether retrieval can work at all.

5. **Always print the baseline next to the accuracy.** "Always guess the majority" is the number
   every model must beat. An accuracy figure without its baseline is not a result.

6. **Make each modality justify itself — then believe the measurement.** We used the photograph and
   then ablated it, and on this data the image is doing real work: two fraudulent claims have
   immaculate paperwork and are caught by nothing but the picture contradicting the form. That is the
   case for multimodality stated properly — not "more inputs are better", but *this question could
   not be asked of the document alone*.

7. **Two independent assessors are worth more than one better one.** The hand-written checks and the
   precedent vote agree on most claims, and where they agree they are right. Their disagreements are
   not a problem to average away — they are the shortlist of claims that deserve a human.

8. **A triage system's output is a decision plus a paper trail.** Not a score. The adjuster has to be
   able to disagree with it, line by line, or they will stop using it.

---

### Things to try

- Change `NEW_CLAIM` to a different held-out claim and re-run Part 5. Do the flags still tell a
  coherent story? Is the retrieved precedent still one you would accept?
- Use the dropdown in Part 4 to query a claim you think is unambiguous, then one you think is
  borderline. Does the neighbourhood change the way you expected?
- Drop `photo_severity` from `FEATURE_NAMES` and re-run Part 4. What changes, and what does that
  tell you about the image's contribution to *retrieval* specifically?
- `red_flags` hard-codes a 35% threshold for `large_claim_vs_cover`. Who should own that number in a
  real insurer — the data team, or the claims department? What happens to complaint volumes if it
  moves to 25%?
- The `police_report_not_filed` check is unusually predictive on this dataset. Look at how often it
  fires and on which claims: is it doing real work, or has it memorised an artefact of how these
  examples were written? How would you tell the difference with real claims?
- Every severe-looking photograph in this book belongs to a fraudulent claim — there is no honest
  total loss in the data. What would you want to see before trusting `photo_worse_than_described` on
  real claims, and why is its absence a property of the dataset rather than evidence the check works?
